In [ ]:
import os
from groq import Groq
import dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
import json
from openai import OpenAI
from openai import embeddings

In [12]:
dotenv.load_dotenv('/mnt/data1tb/datn/.env') 
groq_api_key = os.getenv("GROQ_API_KEY")
open_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=open_api_key)
# client = Groq(api_key=groq_api_key)
# MODEL = 'llama-3.3-70b-versatile'
MODEL = 'gpt-4o'

In [13]:
# Khởi tạo vector database với FAISS
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={'device': 'cpu'}
)

In [14]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "rag_aio",
            "description": "Trả lời câu hỏi dựa trên dữ liệu từ tài liệu AIO PDF (ví dụ: AI Agent là gì)",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "Câu hỏi của người dùng cần trả lời"
                    }
                },
                "required": ["question"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "rag_billionares",
            "description": "Trả lời câu hỏi dựa trên dữ liệu từ tài liệu Billionares PDF (ví dụ: Có bao nhiêu tỷ phú ở Mỹ)",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "Câu hỏi của người dùng về tỷ phú"
                    }
                },
                "required": ["question"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "rag_economic",
            "description": "Trả lời câu hỏi dựa trên dữ liệu từ tài liệu Economic PDF (về kinh tế và tài chính)",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "Câu hỏi của người dùng về kinh tế"
                    }
                },
                "required": ["question"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_an_appointment",
            "description": "Đặt lịch khám bệnh cho bệnh nhân với các thông tin cần thiết.",
            "parameters": {
                "type": "object",
                "properties": {
                    "phone_number": {
                        "type": "string",
                        "description": "Số điện thoại liên hệ"
                    },
                    "date": {
                        "type": "string",
                        "description": "Ngày khám (DD-MM-YYYY)"
                    },
                    "time": {
                        "type": "string",
                        "description": "Giờ khám (HH:MM)"
                    },
                    "specialty": {
                        "type": "string",
                        "description": "Chuyên khoa cần khám (VD: Nội tổng quát, Tim mạch, Da liễu,...)"
                    },
                    "doctor": {
                        "type": "string",
                        "description": "Bác sĩ mong muốn (có thể để trống nếu không có yêu cầu)"
                    }
                },
                "required": ["full_name", "phone_number", "date", "time", "specialty"]
            }
        }
    }
]

In [15]:
from sentence_transformers import CrossEncoder


# rerank search results
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def retrieve_and_re_rank_advanced(vector_db, query, k=10):
    # Lấy kết quả từ Vector Database
    docs_with_scores = vector_db.similarity_search_with_score(query, k=k)
    
    # Chuẩn bị dữ liệu cho Cross-Encoder
    doc_texts = [doc.page_content for doc, _ in docs_with_scores]
    pairs = [[query, doc] for doc in doc_texts]
    
    # Dùng Cross-Encoder để đánh giá lại
    rerank_scores = cross_encoder.predict(pairs)
    
    # Sắp xếp lại theo score mới
    ranked_docs = sorted(zip(doc_texts, rerank_scores), key=lambda x: x[1], reverse=True)
    
    results = [doc for doc, _ in ranked_docs]
    scores = [score for _, score in ranked_docs]
    
    return results, scores


In [16]:
def book_an_appointment(phone_number=None, date=None, time=None, specialty=None, doctor=None):
    required_fields = ["phone_number", "date", "time", "specialty"]
    missing_fields = [field for field in required_fields if not locals()[field]]

    if missing_fields:
        return {
            "status": "incomplete",
            "missing_fields": missing_fields
        }

    return {
        "status": "success",
        "appointment_info": {
            "phone_number": phone_number,
            "date": date,
            "time": time,
            "specialty": specialty,
            "doctor": doctor or "Không yêu cầu bác sĩ cụ thể"
        }
    }


In [17]:
def rag_aio(question: str):
    vector_db = FAISS.load_local('/home/thangcn/Downloads/datn/faiss_db/pdf_aio', embeddings, allow_dangerous_deserialization=True)
    retrieved_docs, scores = retrieve_and_re_rank_advanced(vector_db, question)
        # Gửi yêu cầu đến mô hình Groq
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a helpful assistant that utilizes retrieved information."},
            {"role": "user", "content": f"Context:\n{retrieved_docs[0]}\n\nQuestion: {question}"},
        ],
        model= MODEL,
    )
    # print(chat_completion.choices[0].message.content)
    return chat_completion.choices[0].message.content

def rag_billionares(question: str):
    vector_db = FAISS.load_local('/home/thangcn/Downloads/datn/faiss_db/pdf_billionares', embeddings, allow_dangerous_deserialization=True)
    retrieved_docs, scores = retrieve_and_re_rank_advanced(vector_db, question)
        # Gửi yêu cầu đến mô hình Groq
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a helpful assistant that utilizes retrieved information."},
            {"role": "user", "content": f"Context:\n{retrieved_docs[0]}\n\nQuestion: {question}"},
        ],
        model= MODEL,
    )
    # print(chat_completion.choices[0].message.content)
    return chat_completion.choices[0].message.content

def rag_economic(question: str):
    vector_db = FAISS.load_local('/home/thangcn/Downloads/datn/faiss_db/pdf_economic', embeddings, allow_dangerous_deserialization=True)
    retrieved_docs, scores = retrieve_and_re_rank_advanced(vector_db, question)
        # Gửi yêu cầu đến mô hình Groq
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a helpful assistant that utilizes retrieved information."},
            {"role": "user", "content": f"Context:\n{retrieved_docs[0]}\n\nQuestion: {question}"},
        ],
        model= MODEL,
    )
    # print(chat_completion.choices[0].message.content)
    return chat_completion.choices[0].message.content

def rag_medical(question: str):
    vector_db = FAISS.load_local('/home/thangcn/Downloads/datn/faiss_db/pdf_medical', embeddings, allow_dangerous_deserialization=True)
    retrieved_docs, scores = retrieve_and_re_rank_advanced(vector_db, question)
        # Gửi yêu cầu đến mô hình Groq
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a helpful assistant that utilizes retrieved information."},
            {"role": "user", "content": f"Context:\n{retrieved_docs[0]}\n\nQuestion: {question}"},
        ],
        model= MODEL,
    )
    # print(chat_completion.choices[0].message.content)
    return chat_completion.choices[0].message.content

In [18]:
def run_conversation(user_prompt):
    messages = [
        {
            "role": "system",
            "content": 'Xin chào! Tôi là trợ lý AI được vận hành bởi công nghệ RAG. Tôi chỉ có thể cung cấp thông tin dựa trên những gì tôi truy xuất được từ cơ sở kiến thức của mình. Nếu không tìm thấy thông tin liên quan trong cơ sở dữ liệu, tôi sẽ thông báo bằng cách nói "Tôi không được cung cấp thông tin về chủ đề này." Điều này đảm bảo rằng câu trả lời của tôi được đặt trên nền tảng dữ liệu thực tế thay vì tạo ra thông tin không có nguồn thích hợp.'
        },
        {
            "role": "user",
            "content": user_prompt,
        }
    ]

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        max_tokens=4096
    )
    print(response)
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    if tool_calls:
        # try:
        available_functions = {
            "rag_aio": rag_aio,
            "rag_billionares": rag_billionares,
            "rag_economic": rag_economic,
            "rag_medical": rag_medical,
            "book_an_appointment": book_an_appointment
        }

        messages.append(response_message)
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            print(function_name)
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            function_response = function_to_call(**function_args)
            
            return function_response
   
    else:
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            max_tokens=4096
        )
        messages.append(response_message)
        final_response = response.choices[0].message.content

        return final_response

In [20]:
input = "Tôi muốn đặt một lịch hẹn khám, sdt 0332510038,5h chieu ngay 11/3/2025, khoa Tai Mui Hong, "
response = run_conversation(input)
print(response)

ChatCompletion(id='chatcmpl-B9ieW3kffgpR10LbQYK3FrGFI5SUe', choices=[Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_dbBEdEZ1pkAVWOEuJGUeR3cW', function=Function(arguments='{"phone_number":"0332510038","date":"11-03-2025","time":"17:00","specialty":"Tai Mui Hong"}', name='book_an_appointment'), type='function')]))], created=1741656296, model='gpt-4o-2024-08-06', object='chat.completion', service_tier='default', system_fingerprint='fp_f9f4fb6dbf', usage=CompletionUsage(completion_tokens=43, prompt_tokens=444, total_tokens=487, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))
book_an_appointment


KeyError: 'phone_number'

In [ ]:
python from llama import LlamaIndex, VectorStoreIndex 
from llama_tools import QueryEngineTool, FunctionTool 
from llama_agent import OpenAIAgent 
# Import thư viện cần thiết 
import numpy as np 
# # Định nghĩa hàm multiply 
def multiply(a: int, b: int) -> int: 
    """ Hàm nhân hai số nguyên. 
    Args: 
    a (int): Số nguyên đầu tiên 
    b (int): Số nguyên thứ hai 
    Returns: int: Kết quả phép nhân của a và b 
    """ 
    return a * b 
# # Định nghĩa hàm save_result 
def save_result(result: int) -> str: 
    """ 
    Hàm lưu kết quả vào tệp result.txt. 
    Args: result (int): 
    Kết quả cần lưu Returns: str: 
    Thông báo xác nhận đã lưu kết quả 
    """ 
    with open("result.txt", "w") as f: f.write(str(result)) 
    return "Kết quả đã được lưu thành công." 
# # Tạo tài liệu và index 
text = "Mèo Ú là một giống mèo nội địa của Úc. Chúng được biết đến với bộ lông mềm mại và tính cách thân thiện." 
doc = {"text": text} 
index = VectorStoreIndex([doc]) 
# # Tạo công cụ QueryEngineTool 
query_tool = QueryEngineTool(index) 
# # Tạo công cụ FunctionTool 
multiply_tool = FunctionTool(multiply) 
save_result_tool = FunctionTool(save_result) 
# # Tạo OpenAIAgent 
agent = OpenAIAgent( 
    tools=[query_tool, multiply_tool, save_result_tool], 
    prompt="Tôi sẽ giúp bạn tìm kiếm thông tin về mèo Ú và thực hiện các phép tính cơ bản." 
) 
# # Test agent 
print(agent.query("Mèo Ú là gì?")) 
print(agent.function("multiply", 5, 7)) 
print(agent.function("save_result", 10)) 